## Reconnaissance de la main du graveur — embeddings geles (DINOv2 vs SigLIP)

Objectif : tester si des embeddings d'image pre-entraines (sans fine-tuning) separent deja
les graveurs par leur style, avant d'investir dans un entrainement complet. Limite aux 9
graveurs ayant au moins 100 illustrations segmentees (voir `01_constitution_dataset.ipynb`)
— les 13 autres n'ont pas assez d'exemples pour un split train/test qui veuille dire quelque
chose.

**Point methodologique important** : 3 de ces 9 graveurs (Tempesta, Baur, De Passe) ont deux
editions segmentees chacun — on peut donc tester la generalisation a une edition jamais vue,
le seul test vraiment honnete de "reconnait-il la main". Les 6 autres n'ont qu'une seule
edition : le split ne peut se faire qu'au niveau image, donc une bonne precision sur ces
classes peut en partie refleter des artefacts de scan/papier propres a l'edition plutot que
la main du graveur — resultats a interpreter avec prudence, presentes a part.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from scipy.stats import binomtest

RACINE = Path("../../").resolve()
SEG_DIR = RACINE / "data" / "editions_ovide" / "segmentees"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device :", DEVICE)


Device : cuda


### 1. Dataset filtre — 9 graveurs (>=100 illustrations)

In [2]:
# Sous-ensemble de GRAVEURS (01_constitution_dataset.ipynb), limite aux graveurs avec
# >=100 illustrations segmentees.
GRAVEURS_RETENUS = {
    "tempesta": ["cuivre_tempesta_dejode_anvers1606", "cuivre_tempesta_jansonius_amsterdam1610"],
    "baur":     ["cuivre_baur_sn_augsbourg1709", "cuivre_baur_sn_vienne1639"],
    "de_passe": ["cuivre_depasse_depasse_koln1602", "cuivre_depasse_jansonius_arnhem1607"],
    "solis":    ["bois_solis_feyerabend_francfort1581"],
    "borcht":   ["cuivre_borcht_plantin_anvers1591"],
    "salomon":  ["bois_salomon_rouille_lyon1557"],
    "monconet": ["cuivre_monconet_sommaville_paris1660"],
    "mathieu":  ["cuivre_mathieu_langelier_paris1619"],
    "bouche":   ["cuivre_bouche_blaeu_amsterdam1702"],
}

# graveurs avec plusieurs editions segmentees -> split par edition possible
GRAVEURS_MULTI_EDITIONS = {g: d for g, d in GRAVEURS_RETENUS.items() if len(d) > 1}
print(f"{len(GRAVEURS_RETENUS)} graveurs retenus, dont {len(GRAVEURS_MULTI_EDITIONS)} "
      f"avec plusieurs editions : {list(GRAVEURS_MULTI_EDITIONS)}")

illustrations = []
for graveur, dossiers in GRAVEURS_RETENUS.items():
    for dossier in dossiers:
        fichiers = sorted(f for f in (SEG_DIR / dossier).glob("*.jpg") if "_flip" not in f.name)
        for f in fichiers:
            illustrations.append({"chemin": f, "graveur": graveur, "dossier": dossier})

print(f"\n{len(illustrations)} illustrations au total")
for graveur, dossiers in GRAVEURS_RETENUS.items():
    n = sum(1 for i in illustrations if i["graveur"] == graveur)
    print(f"  {graveur:12s} : {n:4d}  ({', '.join(dossiers)})")


9 graveurs retenus, dont 3 avec plusieurs editions : ['tempesta', 'baur', 'de_passe']

1780 illustrations au total
  tempesta     :  288  (cuivre_tempesta_dejode_anvers1606, cuivre_tempesta_jansonius_amsterdam1610)
  baur         :  286  (cuivre_baur_sn_augsbourg1709, cuivre_baur_sn_vienne1639)
  de_passe     :  270  (cuivre_depasse_depasse_koln1602, cuivre_depasse_jansonius_arnhem1607)
  solis        :  184  (bois_solis_feyerabend_francfort1581)
  borcht       :  182  (cuivre_borcht_plantin_anvers1591)
  salomon      :  161  (bois_salomon_rouille_lyon1557)
  monconet     :  148  (cuivre_monconet_sommaville_paris1660)
  mathieu      :  135  (cuivre_mathieu_langelier_paris1619)
  bouche       :  126  (cuivre_bouche_blaeu_amsterdam1702)


### 2. Chargement des deux extracteurs (DINOv2, SigLIP)

In [3]:
from transformers import AutoImageProcessor, AutoModel

processeur_dino = AutoImageProcessor.from_pretrained("facebook/dinov2-base")
modele_dino = AutoModel.from_pretrained("facebook/dinov2-base").to(DEVICE).eval()

processeur_siglip = AutoImageProcessor.from_pretrained("google/siglip-base-patch16-224")
modele_siglip = AutoModel.from_pretrained("google/siglip-base-patch16-224").to(DEVICE).eval()


def embed_dino(chemin):
    # niveaux de gris puis re-RGB : la coloration est faite par un enlumineur/possesseur
    # du livre apres coup, pas par le graveur -- la garder introduirait un signal trompeur.
    img = Image.open(chemin).convert("RGB").convert("L").convert("RGB")
    x = processeur_dino(images=img, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        sortie = modele_dino(**x)
    return sortie.pooler_output[0].cpu().numpy()


def embed_siglip(chemin):
    img = Image.open(chemin).convert("RGB").convert("L").convert("RGB")
    x = processeur_siglip(images=img, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        vec = modele_siglip.get_image_features(**x).pooler_output
    return vec[0].cpu().numpy()


print("DINOv2 et SigLIP charges")


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

DINOv2 et SigLIP charges


### 3. Calcul des embeddings (les deux modeles, toutes les illustrations)

In [4]:
def calculer_embeddings(chemins, fonction_embed):
    vecteurs = []
    for k, chemin in enumerate(chemins, 1):
        print(f"  {k}/{len(chemins)}", end="\r")
        vecteurs.append(fonction_embed(chemin))
    print()
    return np.array(vecteurs)


chemins = [i["chemin"] for i in illustrations]
y_graveur = np.array([i["graveur"] for i in illustrations])
y_dossier = np.array([i["dossier"] for i in illustrations])

print("Embeddings DINOv2...")
X_dino = calculer_embeddings(chemins, embed_dino)
print("Embeddings SigLIP...")
X_siglip = calculer_embeddings(chemins, embed_siglip)
print(X_dino.shape, X_siglip.shape)


Embeddings DINOv2...


  1780/1780
Embeddings SigLIP...


  1780/1780
(1780, 768) (1780, 768)


### 4. Protocole d'evaluation

Deux tests distincts, voir la remarque en tete de notebook :
- **Split par edition** (Tempesta, Baur, De Passe) : la 1ere edition (ordre alphabetique)
  sert de reference, la 2e — jamais vue — sert de test. Les 6 graveurs a edition unique
  restent dans la reference (jamais testes ici), pour que le test reste une vraie tache a
  9 classes et pas seulement a 3.
- **Split image aleatoire stratifie** (les 6 graveurs a edition unique) : a interpreter avec
  prudence.

In [5]:
def construire_split_edition(graveurs_multi, y_graveur, y_dossier):
    masque_ref, masque_test = [], []
    for g, d in zip(y_graveur, y_dossier):
        if g in graveurs_multi:
            editions = sorted(graveurs_multi[g])
            masque_ref.append(d == editions[0])
            masque_test.append(d == editions[1])
        else:
            masque_ref.append(True)
            masque_test.append(False)
    return np.array(masque_ref), np.array(masque_test)


masque_ref, masque_test = construire_split_edition(GRAVEURS_MULTI_EDITIONS, y_graveur, y_dossier)
print(f"Reference : {masque_ref.sum()} illustrations   Test (edition jamais vue) : {masque_test.sum()}")
print(pd.Series(y_graveur[masque_test]).value_counts())


Reference : 1370 illustrations   Test (edition jamais vue) : 410
tempesta    149
de_passe    136
baur        125
Name: count, dtype: int64


### 5. Precision 1-plus-proche-voisin — test par edition

In [6]:
def precision_1ppv(X_ref, y_ref, X_test, y_test):
    sim = cosine_similarity(X_test, X_ref)
    voisins = sim.argmax(axis=1)
    predictions = y_ref[voisins]
    corrects = predictions == y_test
    return corrects.mean(), corrects, predictions


def rapport_par_graveur(y_test, corrects):
    df = pd.DataFrame({"graveur": y_test, "correct": corrects})
    return df.groupby("graveur")["correct"].agg(["mean", "count"]).rename(
        columns={"mean": "precision", "count": "n_test"})


prec_dino, corrects_dino, pred_dino = precision_1ppv(
    X_dino[masque_ref], y_graveur[masque_ref], X_dino[masque_test], y_graveur[masque_test])
prec_siglip, corrects_siglip, pred_siglip = precision_1ppv(
    X_siglip[masque_ref], y_graveur[masque_ref], X_siglip[masque_test], y_graveur[masque_test])

print(f"DINOv2  : precision globale {prec_dino:.1%}  (hasard = 1/9 = {1/9:.1%})")
print(rapport_par_graveur(y_graveur[masque_test], corrects_dino))
print(f"\nSigLIP  : precision globale {prec_siglip:.1%}")
print(rapport_par_graveur(y_graveur[masque_test], corrects_siglip))


DINOv2  : precision globale 90.7%  (hasard = 1/9 = 11.1%)
          precision  n_test
graveur                    
baur       0.968000     125
de_passe   0.992647     136
tempesta   0.778523     149

SigLIP  : precision globale 89.5%
          precision  n_test
graveur                    
baur       0.960000     125
de_passe   0.992647     136
tempesta   0.751678     149


### 6. Precision 1-plus-proche-voisin — split image aleatoire (graveurs a edition unique)

In [7]:
GRAVEURS_UNIQUES = [g for g in GRAVEURS_RETENUS if g not in GRAVEURS_MULTI_EDITIONS]
masque_uniques = np.isin(y_graveur, GRAVEURS_UNIQUES)

idx_uniques = np.where(masque_uniques)[0]
idx_train, idx_test = train_test_split(
    idx_uniques, test_size=0.3, random_state=42, stratify=y_graveur[idx_uniques])

for nom, X in [("DINOv2", X_dino), ("SigLIP", X_siglip)]:
    prec, corrects, _ = precision_1ppv(X[idx_train], y_graveur[idx_train], X[idx_test], y_graveur[idx_test])
    print(f"{nom:8s}: precision {prec:.1%}  (hasard = 1/{len(GRAVEURS_UNIQUES)} = {1/len(GRAVEURS_UNIQUES):.1%})")


DINOv2  : precision 68.3%  (hasard = 1/6 = 16.7%)
SigLIP  : precision 74.0%  (hasard = 1/6 = 16.7%)


### 7. Test de McNemar — DINOv2 vs SigLIP

Sur le test par edition (le seul fiable) : compare les deux modeles sur les memes images,
ne regarde que celles ou ils sont en desaccord.

In [8]:
def mcnemar(correctsA, correctsB, nomA, nomB):
    b = int(np.sum(correctsA & ~correctsB))
    c = int(np.sum(~correctsA & correctsB))
    print(f"{nomA} correct / {nomB} faux : {b}    {nomB} correct / {nomA} faux : {c}")
    if b + c == 0:
        print("Aucun desaccord — modeles identiques sur cet echantillon.")
        return
    p = binomtest(min(b, c), b + c, 0.5).pvalue
    print(f"p-value (test binomial exact) : {p:.4f}")
    print("Ecart significatif (p<0.05)" if p < 0.05 else "Ecart non significatif : peut etre du bruit d'echantillonnage")


mcnemar(corrects_dino, corrects_siglip, "DINOv2", "SigLIP")


DINOv2 correct / SigLIP faux : 12    SigLIP correct / DINOv2 faux : 7
p-value (test binomial exact) : 0.3593
Ecart non significatif : peut etre du bruit d'echantillonnage


### 8. Sonde lineaire (regression logistique) — meme protocole par edition

In [9]:
for nom, X in [("DINOv2", X_dino), ("SigLIP", X_siglip)]:
    clf = LogisticRegression(max_iter=2000, C=1.0)
    clf.fit(X[masque_ref], y_graveur[masque_ref])
    prec = clf.score(X[masque_test], y_graveur[masque_test])
    print(f"{nom:8s}: precision {prec:.1%}")


DINOv2  : precision 94.1%


SigLIP  : precision 81.2%


### 9. Matrice de confusion (test par edition, DINOv2)

In [10]:
ordre = list(GRAVEURS_RETENUS)
confusion = pd.crosstab(
    pd.Series(y_graveur[masque_test], name="reel"),
    pd.Series(pred_dino, name="predit"),
).reindex(index=[g for g in ordre if g in set(y_graveur[masque_test])], columns=ordre, fill_value=0)
print(confusion)


predit    tempesta  baur  de_passe  solis  borcht  salomon  monconet  mathieu  \
reel                                                                            
tempesta       116     0         0      0       0        0        30        3   
baur             0   121         1      0       2        0         1        0   
de_passe         1     0       135      0       0        0         0        0   

predit    bouche  
reel              
tempesta       0  
baur           0  
de_passe       0  


## Bilan

**Signal réel, pas du bruit** : sur le seul test qui compte vraiment (édition jamais vue,
410 illustrations Tempesta/Baur/De Passe tenues à l'écart), DINOv2 atteint 90,7 % de
précision au plus-proche-voisin et 94,1 % avec une sonde linéaire — très loin du hasard
(11,1 % sur 9 classes). Des embeddings gelés, sans aucun fine-tuning, séparent déjà la main
du graveur.

**DINOv2 vs SigLIP** : l'écart en plus-proche-voisin (90,7 % vs 89,5 %) n'est pas
statistiquement significatif (test de McNemar, p=0,36 sur les désaccords) — sur ce test,
les deux se valent. La sonde linéaire creuse davantage l'écart en faveur de DINOv2 (94,1 %
vs 81,2 %), signe que son espace sépare mieux les classes de façon linéaire — mais un seul
run ne suffit pas à trancher définitivement (voir limite ci-dessous).

**Une confusion expliquée, pas une erreur du modèle** : sur les 149 illustrations de
Tempesta tenues à l'écart, 30 (20 %) sont confondues avec Moncornet et 3 avec Mathieu —
jamais dispersées vers les autres graveurs. Vérifié à deux niveaux : le tableau de
référence de Céline (`retours_celine/BNU_corpus.ods`, feuille Synthèse) note explicitement
Moncornet et Mathieu comme **« copie Antonio Tempesta »** ; à l'œil, la première planche de
Moncornet (« La création du monde ») reprend la composition de Tempesta quasi trait pour
trait, inversée en miroir. Ce constat rejoint celui déjà fait côté `visualisations/` sur la
circulation des plaques et les copies entre graveurs — DINOv2 n'invente rien : il détecte
une vraie parenté iconographique documentée. C'est une **validation** du signal plutôt
qu'une faiblesse : le modèle capte une ressemblance stylistique réelle, y compris quand
elle vient d'une copie assumée plutôt que d'une main partagée.

**Limite assumée** : le test sur les 6 graveurs à édition unique (68,3 % DINOv2 / 74,0 %
SigLIP, largement au-dessus du hasard à 16,7 %, mais mesuré sans édition tenue à l'écart)
reste à interpréter avec prudence — une partie de ce score peut refléter des artefacts de
scan/papier propres à l'édition plutôt que la main du graveur.

**Recommandation** : le signal est assez fort pour justifier un fine-tuning (tête MLP sur
DINOv2 gelé, comme fait pour bois/cuivre), avec vérification multi-graines pour s'assurer
que le gain est réel et pas du bruit d'entraînement — même méthodologie que
`classification_bois_cuivre`/`selection_modele.ipynb`.